# HW3: Attention and the Transformer Block

Run every cell from the top. **Everything already works.**

**Out:** Week 7, Class 1 · **Due:** Week 8, Class 1 · **100 points** · individual work

Work through the notebook and fill in each YOUR TURN cell. Submit this
`.ipynb` with all cells run and their output visible. There is no test to
pass: you are graded on the code working and on your short written answers.

The technical heart of the course. Everything is small enough to check
by hand, which is the point: you should be able to verify each step.

Today you will:

1. Implement scaled dot-product attention from the equation.
2. Add a causal mask and prove no position sees the future.
3. Assemble a block and explain why the shape has to survive it.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup.
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
SENTENCE = ["the", "cat", "sat", "on", "the", "mat"]
SEQ, D = len(SENTENCE), 16
x = torch.randn(SEQ, D)
print("input:", tuple(x.shape), "= (words, features)")

## Part 1. Scaled dot-product attention (35 points)

The equation is softmax(QK^T / sqrt(d_k)) V. Write it out.

In [ ]:
# ================== YOUR TURN 1 ==================
# Implement attention(Q, K, V). Return the output AND the weights.
#
# (20 points)
#
# Hint: scores = Q @ K.T / sqrt(d_k); weights = F.softmax(scores, dim=-1)
#
# Expected: output has the same shape as V, (6, 16). Every row of the weights
#           sums to 1.0. If your rows do not sum to 1 you softmaxed the wrong
#           dimension.
# ===============================================
def attention(Q, K, V, mask=None):
    """Returns (output, weights)."""
    return None, None          # <-- your code here

Q = K = V = x
out, weights = attention(Q, K, V)

if out is None:
    print("not yet")
else:
    print("output shape :", tuple(out.shape))
    print("weights shape:", tuple(weights.shape))
    print("rows sum to 1:", torch.allclose(weights.sum(-1), torch.ones(SEQ), atol=1e-5))

In [ ]:
# ================== YOUR TURN 2 ==================
# Show that the sqrt(d_k) division matters. Compare the largest weight
# with and without it, at d_k = 16 and d_k = 512.
#
# (15 points)
#
# Expected: at d_k = 16 the difference is small. At 512 the unscaled version puts
#           almost all the weight on one word, because dot products of random
#           vectors grow like sqrt(d). Explain what that does to the gradient.
# ===============================================
for d_k in (16, 512):
    torch.manual_seed(0)
    q, k, v = torch.randn(SEQ, d_k), torch.randn(SEQ, d_k), torch.randn(SEQ, d_k)
    # <-- compute weights with and without the division, print the max of each
    print(f"d_k = {d_k}: (fill this in)")

# YOUR ANSWER (2 to 3 sentences): what does a saturated softmax do to the
# gradient, and why does that stop a deep model training?
ANSWER_2 = """
"""

## Part 2. Masking (30 points)

A generative model must not read ahead. Prove yours does not.

In [ ]:
# ================== YOUR TURN 3 ==================
# Build a causal mask and pass it to your attention function so that
# position i cannot attend to any position j > i.
#
# (20 points)
#
# Hint: torch.triu(torch.ones(SEQ, SEQ), diagonal=1).bool(), then masked_fill with -inf
#
# Expected: the total weight on future positions is EXACTLY 0.0, not merely
#           small. If you get a small nonzero number you added a large negative
#           number instead of -inf before the softmax.
# ===============================================
mask = None          # <-- build it

out_m, weights_m = attention(Q, K, V, mask=mask)

if weights_m is None:
    print("not yet: attention() must accept and apply the mask")
else:
    future = weights_m.triu(diagonal=1).sum().item()
    print(f"total weight on future positions: {future}")
    print("exactly zero:", future == 0.0)
    plt.figure(figsize=(3.6, 3.2))
    plt.imshow(weights_m.detach(), cmap="Reds")
    plt.xticks(range(SEQ), SENTENCE, rotation=45); plt.yticks(range(SEQ), SENTENCE)
    plt.title("masked attention"); plt.tight_layout(); plt.show()

In [ ]:
# ================== YOUR TURN 4 ==================
# Explain, in your own words, what would go wrong at TRAINING time and
# at GENERATION time if you forgot the mask.
#
# (10 points)
#
# Expected: training loss would look excellent because the model can read the
#           answer; generation would collapse because that information is not
#           there. The gap between the two is the tell.
# ===============================================
# YOUR ANSWER (3 to 4 sentences):
ANSWER_4 = """
"""
print(ANSWER_4.strip() or "not yet")

## Part 3. The block (35 points)

Attention, a feedforward network, two residual connections, two layer norms.

In [ ]:
# ================== YOUR TURN 5 ==================
# Complete the Block class so a (batch, words, features) tensor comes
# back with exactly the same shape.
#
# (25 points)
#
# Expected: input (1, 6, 16) and output (1, 6, 16). Stacking six of them still
#           gives (1, 6, 16), which is the property that makes deep Transformers
#           possible at all.
# ===============================================
class Block(nn.Module):
    def __init__(self, d_model, heads=2):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.ReLU(),
                                nn.Linear(4 * d_model, d_model))
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, h):
        # <-- attention + residual + norm, then feedforward + residual + norm
        return h

xb = x.unsqueeze(0)
block = Block(D)
y = block(xb)
print("in :", tuple(xb.shape))
print("out:", tuple(y.shape))
print("shape preserved:", xb.shape == y.shape)

In [ ]:
# ================== YOUR TURN 6 ==================
# Count the parameters in one block and say where they mostly live.
#
# (10 points)
#
# Expected: the feedforward network holds roughly two thirds of them, because it
#           widens to 4x and back. Attention is cheaper than most people assume.
# ===============================================
total = sum(p.numel() for p in block.parameters())
attn_params = sum(p.numel() for p in block.attn.parameters())
ff_params = sum(p.numel() for p in block.ff.parameters())

print(f"total     : {total:,}")
print(f"attention : {attn_params:,}  ({100 * attn_params / total:.0f}%)")
print(f"feedforward: {ff_params:,}  ({100 * ff_params / total:.0f}%)")

# YOUR ANSWER (1 to 2 sentences): where do the parameters live, and does that
# match what you expected?
ANSWER_6 = """
"""

## Answers

Try each task before reading.

In [ ]:
# Marking
#   Q1  20   attention implemented, shapes right, rows sum to 1
#   Q2  15   both d_k measured; gradient explanation correct
#   Q3  20   mask applied via -inf so future weight is exactly 0
#   Q4  10   both failure modes described
#   Q5  25   block completed, shape preserved
#   Q6  10   parameter counts and one sentence of interpretation
#
# Common mistakes:
#   - softmax(dim=0) instead of dim=-1: the columns sum to 1, not the rows
#   - masking by multiplying by 0 AFTER the softmax, which unnormalises the rows
#   - forgetting the residual, so the block trains far worse and you cannot see why